# W6 — nhánh P1 trên tập **dev** dev_test_v2, Kaggle T4

Notebook chạy **tự động từ đầu tới cuối**, nên dùng được **Save Version → Save & Run All (Commit)**: Kaggle chạy nền, tắt máy cũng được, kết quả nằm trong tab **Output** của version.

**Cài đặt notebook (cột phải → Session options):** Accelerator = **GPU T4 x2** (notebook chỉ dùng 1 GPU); **Internet = On** (tài khoản phải xác minh số điện thoại). Không cần gắn dataset, trừ khi tải checkpoint bị lỗi (xem Ô 4).

| Ô | Làm gì | Ước lượng |
|---|---|---|
| 1–2 | Kiểm GPU, lấy code, **kiểm dev đã LOCKED và hash `dev/inputs.jsonl` khớp** | 1 phút |
| 3 | Môi trường Python 3.10 giống Colab (torch 2.5.1+cu121, transformers 4.44.2) + build CUDA ops | 10–20 phút |
| 4 | Ảnh MUCCA (Zenodo, kiểm md5), checkpoint CountGD++ (kiểm sha256), BERT | 3–5 phút |
| 5 | **Kiểm A:** CLIP cho điểm giống bản đã lưu | 1–2 phút |
| 6 | **Chạy P1 trên dev:** tau 0.18–0.28, M = 10, `--replay-b2` (60 query) | khoảng 15–40 phút |
| 7 | Bảng đối chứng (`p1_controls.py`: điểm CLIP cho mọi nguồn, gồm B0 → C6) + chấm nhanh trên dev | 5–10 phút |
| 8 | **Kiểm C:** chạy lại y hệt Ô 6 và so tuyệt đối | như Ô 6 |
| 9 | Lưới M: chạy thêm M = 5 + bảng đối chứng | như Ô 6–7 |
| 10 | Gói kết quả: `for_repo.zip` (file nhỏ, D10) và thư mục raw đầy đủ | 1 phút |

**Quy tắc:** chỉ đọc `dev_test_v2/dev/`. **Không** mở hay chạy `dev_test_v2/test/` (test chạy một lần 23–24/11, sau khi khóa cấu hình). Model không đọc GT; GT dev chỉ dùng để chấm ở Ô 7. Bảng chấm ở Ô 7 dùng ngưỡng S/F cũ (21/10) nên **chỉ để xem nhanh**; chọn cấu hình theo D9 làm sau bằng `select_offline_v2.py` (ngoại tuyến).

**Nền tảng:** Kaggle T4 thay cho Colab T4 (Triệu chọn 11/11 vì Save Version). Chỉ so các cách **trong cùng một lần chạy**; test sau này phải chạy cùng nền tảng (Kaggle T4).

In [ ]:
# Ô 1 — thiết lập. Đổi cờ ở đây nếu muốn bớt việc (ví dụ thiếu quota GPU).
import datetime, glob, hashlib, json, os, pathlib, re, shlex, shutil, subprocess, sys, zipfile
REPO_URL = "https://github.com/minhtridaoo/zero-shot-counting.git"
REPO_REF = "main"           # muốn chạy lại đúng một bản cũ thì ghi mã commit
TAUS = "0.18 0.20 0.22 0.24 0.26 0.28"
RUN_REPEAT = True           # Ô 8: Kiểm C (chạy lại M = 10 và so tuyệt đối)
RUN_M5 = True               # Ô 9: lưới M, thêm một lần M = 5
STAMP = datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=7))).strftime("%Y%m%d_%H%M")
ROOT = pathlib.Path("/tmp/CS331")             # code + dữ liệu + checkpoint: để ngoài /kaggle/working cho Output gọn
PY = "/tmp/venv310/bin/python"
OUT = pathlib.Path("/kaggle/working/w6_out")  # chỉ thư mục này được Kaggle lưu
OUT.mkdir(parents=True, exist_ok=True)
ENV = os.environ.copy()
ENV.update(CUDA_VISIBLE_DEVICES="0", MPLBACKEND="Agg", PYTHONUNBUFFERED="1")
def sh(cmd, cwd=None):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"Lệnh lỗi (mã {p.returncode}), xem các dòng ngay phía trên. Dừng, không chạy tiếp.")
def file_hash(path, algo="sha256"):
    h = hashlib.new(algo)
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()
gpu = subprocess.check_output("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader", shell=True, text=True)
print(gpu)
assert "T4" in gpu.splitlines()[0], "Cần GPU T4 (Session options > Accelerator > GPU T4 x2). Dừng."
print("Disk trống /tmp:", round(shutil.disk_usage("/tmp").free / 1e9, 1), "GB | stamp", STAMP)

In [ ]:
# Ô 2 — lấy code và kiểm tập dev đã khóa (W5). Không đọc thư mục test.
if not ROOT.exists():
    sh(f"git clone -q {REPO_URL} {ROOT}")
sh(f"git fetch -q origin && git checkout -q {REPO_REF} && (git pull -q --ff-only 2>/dev/null || true)", cwd=ROOT)
COMMIT = subprocess.check_output("git rev-parse HEAD", shell=True, cwd=ROOT, text=True).strip()
print("commit", COMMIT)
DEV = ROOT / "experiments/countgdpp/dev_test_v2"
meta = json.loads((DEV / "selection_meta.json").read_text(encoding="utf-8"))
assert meta["status"] == "LOCKED", "dev_test_v2 chưa LOCKED (W5). Dừng."
for f in ("dev/inputs.jsonl", "dev/manifest.csv"):
    assert file_hash(DEV / f) == meta["locked_file_sha256"][f], f"{f} khác hash đã khóa. Dừng."
DEV_INPUTS = "experiments/countgdpp/dev_test_v2/dev/inputs.jsonl"
DEV_MANIFEST = "experiments/countgdpp/dev_test_v2/dev/manifest.csv"
n = sum(1 for _ in open(ROOT / DEV_INPUTS, encoding="utf-8"))
print(f"dev LOCKED {meta['locked_at']} | {n} query (phải là 60) | hash inputs/manifest khớp")
assert n == 60

In [ ]:
# Ô 3 — môi trường /tmp/venv310 giống Colab đã qua Kiểm A/C (torch 2.5.1+cu121, transformers 4.44.2, numpy 2.2.6, pillow 10.4.0)
# và build CUDA ops (MultiScaleDeformableAttention) cho T4 (sm 7.5). Dùng nvcc 12.x có sẵn của Kaggle nếu có, không thì tải CUDA 12.1.
sh("pip install -q uv")
if not os.path.exists(PY):
    sh("uv venv /tmp/venv310 --python 3.10")
sh(f"uv pip install -q -p {PY} torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121")
sh(f"uv pip install -q -p {PY} transformers==4.44.2 timm==1.0.30 numpy==2.2.6 pillow==10.4.0 scipy termcolor addict yapf==0.40.1"
   " opencv-python-headless pycocotools matplotlib setuptools wheel ninja")
nvccs = sorted(set(filter(None, [shutil.which("nvcc"), *glob.glob("/usr/local/cuda*/bin/nvcc")])))
cuda_home = None
for nv in nvccs:
    if re.search(r"release 12\.", subprocess.run([nv, "--version"], capture_output=True, text=True).stdout):
        cuda_home = str(pathlib.Path(nv).resolve().parent.parent); break
if cuda_home is None:
    cuda_home = "/tmp/cuda-12.1"
    if not os.path.exists(f"{cuda_home}/bin/nvcc"):
        sh("wget -q -O /tmp/cuda121.run https://developer.download.nvidia.com/compute/cuda/12.1.1/local_installers/cuda_12.1.1_530.30.02_linux.run")
        sh(f"sh /tmp/cuda121.run --silent --toolkit --toolkitpath={cuda_home} --override --no-man-page && rm -f /tmp/cuda121.run")
sh(f"{cuda_home}/bin/nvcc --version | tail -1")
sh("apt-get -qq update && apt-get -qq install -y gcc-11 g++-11 > /dev/null")
ENV.update(CUDA_HOME=cuda_home, PATH=f"{cuda_home}/bin:/tmp/venv310/bin:" + ENV["PATH"], CC="gcc-11", CXX="g++-11", TORCH_CUDA_ARCH_LIST="7.5")
sh("rm -rf /tmp/ops_build /tmp/wheels && cp -r " + str(ROOT / "CountGDPlusPlus/models/GroundingDINO/ops") + " /tmp/ops_build")
sh("rm -rf /tmp/ops_build/build /tmp/ops_build/dist /tmp/ops_build/*.egg-info")
sh(f"cd /tmp/ops_build && {PY} setup.py -q bdist_wheel -d /tmp/wheels")
wheel = sorted(p for p in glob.glob("/tmp/wheels/*.whl") if os.path.basename(p).lower().startswith("multiscaledeformableattention-"))[-1]
sh(f"uv pip install -q -p {PY} --reinstall {wheel}")
probe = ("import sys, torch, transformers, numpy, PIL, MultiScaleDeformableAttention; "
         "assert torch.__version__.startswith('2.5.1'), torch.__version__; "
         "print('python', sys.version.split()[0], '| torch', torch.__version__, '| transformers', transformers.__version__, "
         "'| numpy', numpy.__version__, '| pillow', PIL.__version__, '| GPU', torch.cuda.get_device_name())")
sh(f"{PY} -c {shlex.quote(probe)}")
sh(f"uv pip freeze -p {PY} > {OUT}/packages.txt")

In [ ]:
# Ô 4 — ảnh MUCCA 1.0 (Zenodo, kiểm md5), checkpoint CountGD++ (kiểm sha256 đã audit 14/10), BERT.
# Nếu gdown báo quá lượt tải: tạo Kaggle Dataset riêng tư chứa countgd_plusplus.pth (bản trên Drive MyDrive/CS331_colab/),
# gắn vào notebook (Add Input) rồi chạy lại; ô này tự tìm file trong /kaggle/input.
MUCCA = ROOT / "data_external/mucca_zenodo_19231375/MUCCA_1.0"
if not (MUCCA / "images/mcac_61.jpg").exists():
    sh('wget -q -O /tmp/MUCCA_1.0.zip "https://zenodo.org/api/records/19231375/files/MUCCA_1.0.zip/content"')
    md5 = file_hash("/tmp/MUCCA_1.0.zip", "md5")
    assert md5 == "b182e445445dad017493ffb35803f816", f"md5 sai: {md5}"
    MUCCA.mkdir(parents=True, exist_ok=True)
    sh(f"unzip -q -o /tmp/MUCCA_1.0.zip -d {MUCCA}")
print("Số ảnh MUCCA:", len(list((MUCCA / "images").glob("*.jpg"))), "(phải là 200)")
for img, digest in meta["image_sha256"].items():
    if img in meta["dev"]:
        assert file_hash(MUCCA / f"images/{img}.jpg") == digest, f"ảnh {img} khác hash đã khóa"
print("20 ảnh dev khớp hash đã khóa")
CKPT = ROOT / "CountGDPlusPlus/checkpoints/countgd_plusplus.pth"
EXPECTED = json.loads((ROOT / "results/countgdpp/checkpoint_audit_2026-10-14.json").read_text())["checkpoint_sha256"]
if not CKPT.exists():
    local = glob.glob("/kaggle/input/**/countgd_plusplus.pth", recursive=True)
    if local:
        shutil.copy(local[0], CKPT)
    else:
        sh(f"uv pip install -q -p {PY} gdown")
        sh(f'/tmp/venv310/bin/gdown "https://drive.google.com/uc?id=1j6N22TtKu2NVcKpgfrf-sJHGeLDqs9hs" -O {CKPT}')
assert file_hash(CKPT) == EXPECTED, "checkpoint sha256 sai"
print("Checkpoint đúng bản đã audit 14/10.")
BERT = ROOT / "CountGDPlusPlus/checkpoints/bert-base-uncased"
if not any((BERT / n).exists() for n in ("model.safetensors", "pytorch_model.bin")):
    sh(f"{PY} download_bert.py", cwd=ROOT / "CountGDPlusPlus")
print("BERT:", sorted(p.name for p in BERT.iterdir()))

In [ ]:
# Ô 5 — KIỂM A: CLIP trên Kaggle cho điểm giống bản đã lưu (chấm lại khung cũ của p4_unseen_v1, không chỉnh gì).
check = """
import json, sys
from pathlib import Path
from PIL import Image
ROOT = Path("/tmp/CS331"); sys.path.insert(0, str(ROOT / "experiments/countgdpp"))
from pseudo_clip_verifier import ClipVerifier
clip = ClipVerifier("cuda")
diffs = []
for f in sorted((ROOT / "results/countgdpp/p4_unseen_v1_clip_20261027").glob("*_clip_scores.json")):
    r = json.loads(f.read_text(encoding="utf-8"))
    rows = r["rows_original_order"]
    if not rows:
        continue
    with Image.open(ROOT / r["image_path"]) as im:
        image = im.convert("RGB")
    got = clip.score(image, [x["original_box_xyxy"] for x in rows], r["prompt"])
    diffs += [abs(a - x["clip_cosine_target"]) for a, x in zip(got, rows)]
print(f"KIEM A: {len(diffs)} khung, lech lon nhat {max(diffs):.2e}, trung binh {sum(diffs)/len(diffs):.2e}")
assert max(diffs) < 1e-3, "Kiem A KHONG DAT"
"""
pathlib.Path("/tmp/clip_check.py").write_text(check, encoding="utf-8")
sh(f"{PY} /tmp/clip_check.py", cwd=ROOT)

In [ ]:
# Ô 6 — CHẠY P1 TRÊN DEV: M = 10, 6 ngưỡng tau, kèm B2 gốc từ cùng lượt 1. Không có tham chiếu (dev chưa từng chạy).
def p1_run(run_id, m):
    sh(f"{PY} experiments/countgdpp/run_b2_verified_pseudo.py --inputs {DEV_INPUTS} --run-id {run_id}"
       f" --tau-pseudo {TAUS} --m {m} --replay-b2 --check-inputs", cwd=ROOT)  # chỉ kiểm input/checkpoint/code, không tạo thư mục
    sh(f"{PY} experiments/countgdpp/run_b2_verified_pseudo.py --inputs {DEV_INPUTS} --run-id {run_id}"
       f" --tau-pseudo {TAUS} --m {m} --replay-b2", cwd=ROOT)
    cfg = json.loads((ROOT / "results/countgdpp" / run_id / "run_config.json").read_text(encoding="utf-8"))
    assert cfg["status"] == "complete", cfg["status"]
    shutil.copytree(ROOT / "results/countgdpp" / run_id, OUT / run_id)
    print("Xong", run_id, "| outputs", cfg["outputs"], "| GPU", cfg.get("device"))
    return run_id
RUN_M10 = p1_run(f"p1_dev_kaggle_t4_{STAMP}_m10", 10)

In [ ]:
# Ô 7 — BẢNG ĐỐI CHỨNG + CHẤM NHANH TRÊN DEV. p1_controls.py chấm CLIP mọi khung của mọi nguồn (B0 = lượt 1, B2, từng tau):
# cần cho C6 (B0 + lọc) và cho việc chọn lại ngưỡng S/F riêng từng nguồn sau này (select_offline_v2.py, ngoại tuyến).
# Bảng in ra dùng ngưỡng S/F cũ khóa 21/10 => CHỈ ĐỂ XEM NHANH, không phải kết quả chọn cấu hình theo D9.
def controls(run_id):
    ctrl = run_id + "_controls"
    sh(f"{PY} experiments/countgdpp/p1_controls.py --run results/countgdpp/{run_id} --run-id {ctrl}", cwd=ROOT)
    sh(f"{PY} evaluation/evaluate_research.py --controls results/countgdpp/{ctrl} --manifest {DEV_MANIFEST}"
       f" --annotations {MUCCA}/annotations.json --run-id {ctrl}_eval", cwd=ROOT)
    for d in (ctrl + "_clip", ctrl, ctrl + "_eval"):
        shutil.copytree(ROOT / "results/countgdpp" / d, OUT / d)
    summary = json.loads((ROOT / "results/countgdpp" / (ctrl + "_eval") / "metrics.json").read_text(encoding="utf-8"))["summary"]
    print(f"\n{run_id} (ngưỡng S/F cũ 21/10, chỉ xem nhanh)")
    print(f"{'nguồn':<14}{'MAE cả tập':>11}{'MAE có mặt':>11}{'FPR_absent':>11}{'TB đếm absent':>14}")
    for src, s in summary.items():
        ab = s["absent"] or dict(FPR_absent=float("nan"), MeanAbsentCount=float("nan"))
        print(f"{src:<14}{s['all']['MAE']:>11.2f}{s['present']['MAE']:>11.2f}{ab['FPR_absent']:>11.2f}{ab['MeanAbsentCount']:>14.2f}")
controls(RUN_M10)

In [ ]:
# Ô 8 — KIỂM C: chạy lại y hệt Ô 6 và so tuyệt đối từng query, mọi cách (lượt 1, mẫu chọn, khung và điểm cuối).
KEYS = ("stage1_boxes", "stage1_scores", "pseudo_proposal_ids", "final_boxes", "final_scores")
def jl(path):
    return {r["query_id"]: r for r in map(json.loads, open(path, encoding="utf-8"))}
if RUN_REPEAT:
    RUN_REP = p1_run(f"p1_dev_kaggle_t4_{STAMP}_m10_rep", 10)
    report = {}
    for v in json.loads((OUT / RUN_M10 / "run_config.json").read_text(encoding="utf-8"))["outputs"]:
        a, b = jl(OUT / RUN_M10 / f"predictions_{v}.jsonl"), jl(OUT / RUN_REP / f"predictions_{v}.jsonl")
        bad = sorted(q for q in a if any(a[q][k] != b[q][k] for k in KEYS))
        report[v] = dict(match=len(a) - len(bad), total=len(a), mismatched=bad)
        print(f"KIỂM C {v:<10}: khớp tuyệt đối {len(a) - len(bad)}/{len(a)}" + (f" | lệch: {bad}" if bad else ""))
    (OUT / f"kiem_c_{STAMP}.json").write_text(json.dumps(dict(run=RUN_M10, repeat=RUN_REP, keys=KEYS, report=report), indent=2), encoding="utf-8")
    print("KIỂM C", "ĐẠT" if all(not r["mismatched"] for r in report.values()) else "KHÔNG ĐẠT: chưa được dùng số dev")

In [ ]:
# Ô 9 — LƯỚI M: thêm một lần M = 5 (cùng 6 tau) + bảng đối chứng. Tắt bằng RUN_M5 = False ở Ô 1 nếu thiếu quota.
if RUN_M5:
    RUN_M5_ID = p1_run(f"p1_dev_kaggle_t4_{STAMP}_m5", 5)
    controls(RUN_M5_ID)

In [ ]:
# Ô 10 — GÓI KẾT QUẢ. for_repo_<stamp>.zip chỉ gồm file D10 cho phép (json/jsonl/csv, KHÔNG .npz, KHÔNG ảnh) để đưa vào repo;
# thư mục w6_out/ giữ đủ raw .npz + ảnh (chia sẻ cho Trí qua notebook/Output, không push).
allowed = {".json", ".jsonl", ".csv", ".txt"}
zip_path = pathlib.Path(f"/kaggle/working/for_repo_{STAMP}.zip")
listing = []
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob("*")):
        if p.is_file():
            listing.append(dict(path=str(p.relative_to(OUT)), bytes=p.stat().st_size, sha256=file_hash(p)))
            if p.suffix in allowed:
                z.write(p, p.relative_to(OUT))
(OUT / f"FILES_{STAMP}.json").write_text(json.dumps(dict(commit=COMMIT, stamp=STAMP, files=listing), indent=2), encoding="utf-8")
with zipfile.ZipFile(zip_path, "a") as z:
    z.write(OUT / f"FILES_{STAMP}.json", f"FILES_{STAMP}.json")
raw = sum(1 for x in listing if x["path"].endswith(".npz"))
print(f"{zip_path.name}: {round(zip_path.stat().st_size / 1e6, 1)} MB | w6_out: {len(listing)} file, {raw} raw .npz,"
      f" {round(sum(x['bytes'] for x in listing) / 1e6, 1)} MB | commit {COMMIT[:7]}")
print("XONG W6. Tải for_repo_*.zip (tab Output của version) gửi assistant; raw để trên Kaggle, chia sẻ notebook cho Trí.")

## Sau khi chạy xong

1. Mở version vừa chạy → tab **Output** → tải **`for_repo_<stamp>.zip`** về máy (thư mục Downloads) và báo assistant. Assistant kiểm hash theo `FILES_<stamp>.json`, rồi chỉ chép file D10 cho phép vào `results/countgdpp/`.
2. Dán (hoặc chụp) phần in ra của **Ô 5 (Kiểm A)**, **Ô 7**, **Ô 8 (Kiểm C)** và **Ô 10**. Kiểm C chưa đạt 60/60 ở mọi cách thì **chưa được dùng số dev**.
3. Raw `.npz` và ảnh nằm trong `w6_out/` của Output. Chia sẻ notebook cho Trí (**Share → Add collaborator**) để Trí/assistant tính T và P+T ngoại tuyến.
4. **Không chạy test.** Chọn cấu hình theo D9 làm ngoại tuyến bằng `select_offline_v2.py` sau khi có output.